# Helper para rodar o corretor automático 

Este notebook monta e executa o comando:

```bash
java -jar ARG1 ARG2 ARG3 ARG4 ARG5 ARG6 ARG7
```

In [9]:
from pathlib import Path
import subprocess
import shutil
import os
import re
from textwrap import indent


## 1. Configure os caminhos

Edite os valores abaixo para a sua máquina.

Use caminhos absolutos. No Windows, prefira `r"C:\\..."` para evitar problemas com barras invertidas.


In [10]:
# ARG1: caminho do .jar do corretor automático
CORRETOR_JAR = r"C:\Users\Usuario\Desktop\compiladores\compiladores-corretor-automatico-1.0-SNAPSHOT-jar-with-dependencies.jar"

# ARG2: comando executável do seu compilador.
# IMPORTANTE: para Java, coloque "java -jar ...", não coloque só o caminho do .jar.
# Se o caminho do jar tiver espaços, use aspas internas:
# COMPILADOR_CMD = r'java -jar "C:\Meu Compilador\meu-compilador.jar"'

#Trabalho 1
COMPILADOR_CMD = r"java -jar C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T1\demo\target\demo-1.0-SNAPSHOT.jar"

#Trabalho 2 antlr
COMPILADOR_CMD = r"java -jar C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T2\t2-la-antlr\target\t2-la-antlr-1.0-SNAPSHOT.jar"

#Trabalho 2 manual
COMPILADOR_CMD = r"java -jar C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T2\t2-la-manual\target\t2-la-manual-1.0-SNAPSHOT.jar"




# ARG3: GCC. Pode ser "gcc" se estiver no PATH, ou o caminho absoluto do gcc.exe.
GCC_CMD = r"C:\mingw64\bin\gcc.exe"

# ARG4: pasta temporária onde o corretor criará as saídas produzidas
TEMP_DIR = r"C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T1\temp"

# ARG5: pasta onde os casos de teste foram descompactados
# Normalmente é a pasta que contém subpastas como 1.casos_teste_t1, 2.casos_teste_t2 etc.
CASOS_TESTE_DIR = r"C:\Users\Usuario\Desktop\compiladores\casos_de_teste"

# ARG6: RAs do grupo
RAS = "799714"

OPCAO = "t1|t2"

# Comando Java. Use "java" se estiver no PATH.
JAVA_CMD = "java"


## 2. Funções auxiliares

Estas funções validam os caminhos, imprimem o comando final e executam o corretor.


In [11]:
def quote_arg(arg: str) -> str:
    """Formata um argumento para exibição no terminal."""
    arg = str(arg)
    if not arg:
        return '""'
    if any(ch.isspace() for ch in arg) or any(ch in arg for ch in ['"', "'"]):
        return '"' + arg.replace('"', '\\"') + '"'
    return arg


def print_command(cmd):
    print("Comando montado:")
    print(" ".join(quote_arg(x) for x in cmd))


def command_exists(cmd: str) -> bool:
    """Checa se um executável existe por caminho absoluto ou pelo PATH."""
    p = Path(cmd)
    if p.exists():
        return True
    return shutil.which(cmd) is not None


def extract_java_jar_path(compilador_cmd: str):
    """
    Tenta extrair o caminho do .jar quando COMPILADOR_CMD é do tipo:
    java -jar C:\\x\\y.jar
    java -jar "C:\\x com espaco\\y.jar"
    """
    pattern = r"java\s+-jar\s+(?:\"([^\"]+\.jar)\"|([^\s]+\.jar))"
    m = re.search(pattern, compilador_cmd, flags=re.IGNORECASE)
    if not m:
        return None
    return m.group(1) or m.group(2)



def montar_comando_corretor():
    return [
        JAVA_CMD,
        "-jar",
        CORRETOR_JAR,
        COMPILADOR_CMD,
        GCC_CMD,
        TEMP_DIR,
        CASOS_TESTE_DIR,
        RAS,
        OPCAO,
    ]


def rodar_corretor():

    cmd = montar_comando_corretor()
    print()
    print_command(cmd)
    print("\nExecutando corretor...\n")

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print("STDOUT")
    print("=" * 32)
    print(result.stdout)

    if result.stderr.strip():
        print("\nSTDERR")
        print("=" * 32)
        print(result.stderr)

    print(f"\nCódigo de retorno: {result.returncode}")
    return result


## 3. Verifique Java, Maven opcional e o jar do seu compilador

Esta célula não roda o corretor ainda. Ela só ajuda a diagnosticar ambiente.


In [12]:
def testar_comando(cmd):
    print(f"\n$ {' '.join(cmd)}")
    try:
        r = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            encoding="utf-8",
            errors="replace",
        )
        saida = (r.stdout + "\n" + r.stderr).strip()
        print(saida if saida else "Sem saída.")
        print(f"returncode={r.returncode}")
    except Exception as e:
        print(f"ERRO: {e}")


testar_comando([JAVA_CMD, "-version"])
testar_comando([GCC_CMD, "--version"] if Path(GCC_CMD).exists() else [GCC_CMD, "--version"])

jar_compilador = extract_java_jar_path(COMPILADOR_CMD)
print("\nJar do compilador detectado:", jar_compilador)
if jar_compilador:
    print("Existe?", Path(jar_compilador).is_file())



$ java -version
openjdk version "25.0.3" 2026-04-21 LTS
OpenJDK Runtime Environment Temurin-25.0.3+9 (build 25.0.3+9-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.3+9 (build 25.0.3+9-LTS, mixed mode, sharing)
returncode=0

$ C:\mingw64\bin\gcc.exe --version
gcc.exe (x86_64-posix-seh-rev0, Built by MinGW-Builds project) 16.1.0
Copyright (C) 2026 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.
returncode=0

Jar do compilador detectado: C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T2\t2-la-manual\target\t2-la-manual-1.0-SNAPSHOT.jar
Existe? True


## 4. Rodar o corretor

Execute a célula abaixo para rodar a correção.


In [13]:
resultado = rodar_corretor()



Comando montado:
java -jar C:\Users\Usuario\Desktop\compiladores\compiladores-corretor-automatico-1.0-SNAPSHOT-jar-with-dependencies.jar "java -jar C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T2\t2-la-manual\target\t2-la-manual-1.0-SNAPSHOT.jar" C:\mingw64\bin\gcc.exe C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T1\temp C:\Users\Usuario\Desktop\compiladores\casos_de_teste 799714 t1|t2

Executando corretor...

STDOUT
Corrigindo T1 ...
   1-algoritmo_2-2_apostila_LA.txt
Executando: java -jar C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T2\t2-la-manual\target\t2-la-manual-1.0-SNAPSHOT.jar C:\Users\Usuario\Desktop\compiladores\casos_de_teste\1.casos_teste_t1\entrada\1-algoritmo_2-2_apostila_LA.txt C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T1\temp\saidaProduzida\saida_t1\1-algoritmo_2-2_apostila_LA.txt
Execu��o finalizada
   10-algoritmo_5-4_apostila_LA.txt
Executando: java -jar C:\Users\Usuario\Desktop\compiladores\di

## 5. Listar arquivos produzidos pelo corretor

Depois de rodar o corretor, esta célula lista arquivos dentro da pasta temporária.

O corretor costuma renomear arquivos com `ok` ou `erro` no nome.


In [14]:
def listar_saidas_temp(limite=200):
    temp = Path(TEMP_DIR)
    if not temp.exists():
        print(f"Pasta temporária não existe: {temp}")
        return []

    arquivos = [p for p in temp.rglob("*") if p.is_file()]
    arquivos = sorted(arquivos, key=lambda p: str(p).lower())

    print(f"Total de arquivos encontrados: {len(arquivos)}")
    print()

    for p in arquivos[:limite]:
        rel = p.relative_to(temp)
        marcador = ""
        nome = p.name.lower()
        if "erro" in nome:
            marcador = "  <-- POSSÍVEL FALHA"
        elif "ok" in nome:
            marcador = "  <-- OK"
        print(f"{rel}{marcador}")

    if len(arquivos) > limite:
        print(f"\nMostrando apenas {limite} arquivos.")

    return arquivos


arquivos_temp = listar_saidas_temp()


Total de arquivos encontrados: 99

saidaProduzida\saida_t1\_erro_1-algoritmo_2-2_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_10-algoritmo_5-4_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_11-algoritmo_6-1_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_12-algoritmo_6-2_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_13-algoritmo_6-4_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_14-algoritmo_6-5_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_15-algoritmo_6-6_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_16-algoritmo_6-9_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_17-algoritmo_6-10_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_18-algoritmo_7-1_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_19-algoritmo_7-2_apostila_LA.txt  <-- POSSÍVEL FALHA
saidaProduzida\saida_t1\_erro_2-algoritmo_2-3_aposti

## 6. Ler um arquivo produzido

Use esta célula para abrir uma saída produzida pelo corretor.


In [15]:
def ler_arquivo(caminho, max_chars=5000):
    p = Path(caminho)
    if not p.is_file():
        print(f"Arquivo não encontrado: {p}")
        return

    texto = p.read_text(encoding="utf-8", errors="replace")
    print(texto[:max_chars])
    if len(texto) > max_chars:
        print(f"\n... truncado em {max_chars} caracteres ...")


# Exemplo de uso depois de listar os arquivos:
# ler_arquivo(r"C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T1\temp\saidaProduzida\saida_t1\algum_arquivo.txt")


## 7. Comparar saída produzida com saída esperada

Use quando um caso falhar e você quiser ver a diferença linha a linha.


In [16]:
import difflib

def comparar_arquivos(esperado, produzido, contexto=3):
    esperado = Path(esperado)
    produzido = Path(produzido)

    if not esperado.is_file():
        print(f"Arquivo esperado não encontrado: {esperado}")
        return
    if not produzido.is_file():
        print(f"Arquivo produzido não encontrado: {produzido}")
        return

    linhas_esp = esperado.read_text(encoding="utf-8", errors="replace").splitlines()
    linhas_prod = produzido.read_text(encoding="utf-8", errors="replace").splitlines()

    diff = difflib.unified_diff(
        linhas_esp,
        linhas_prod,
        fromfile=f"esperado: {esperado.name}",
        tofile=f"produzido: {produzido.name}",
        lineterm="",
        n=contexto,
    )

    texto = "\n".join(diff)
    print(texto if texto else "Arquivos iguais.")


# Exemplo:
# comparar_arquivos(
#     r"C:\Users\Usuario\Desktop\compiladores\casos_de_teste\1.casos_teste_t1\saida\1-algoritmo_2-2_apostila_LA.txt",
#     r"C:\Users\Usuario\Desktop\compiladores\disciplina_compiladores\T1\temp\saidaProduzida\saida_t1\1-algoritmo_2-2_apostila_LA.txt"
# )


## Diagnóstico rápido de erros comuns

### Erro: `CreateProcess error=193, %1 is not a valid Win32 application`

O corretor está tentando executar um `.jar` diretamente.

Errado:

```txt
COMPILADOR_CMD = r"C:\\...\\demo-1.0-SNAPSHOT.jar"
```

Certo:

```txt
COMPILADOR_CMD = r"java -jar C:\\...\\demo-1.0-SNAPSHOT.jar"
```

### Erro: `JAVA_HOME environment variable is not defined correctly`

Corrija o `JAVA_HOME` para apontar para a pasta raiz do JDK, sem `\\bin`.

Exemplo:

```txt
C:\\Program Files\\Eclipse Adoptium\\jdk-25.0.1.8-hotspot
```

E adicione no `Path`:

```txt
%JAVA_HOME%\\bin
```

### Erro: todos os testes falham mas o programa executa

Abra a pasta temporária e compare a saída produzida com a esperada usando `comparar_arquivos(...)`.
